# 🤖 Machine Learning sur les Vecteurs φ

> *phi-harvest collecte des vecteurs AST anonymisés — ce notebook montre comment les utiliser pour la détection automatique de vulnérabilités.*

Ce notebook démontre :
1. Chargement du corpus JSONL de harvest
2. Clustering des vecteurs φ normalisés
3. Classification supervisée des vulnérabilités
4. Visualisation t-SNE de l’espace φ

**Prérequis** : `pip install phi-complexity[notebooks]`

In [ ]:
import os
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

from phi_complexity.core import PHI, PHI_INV
from phi_complexity.harvest import HarvestEngine

## 1. Collecte de Vecteurs φ

Le moteur `phi-harvest` extrait des vecteurs AST **entièrement anonymisés** :
- Zéro identifiant, zéro code source
- Métriques structurelles pures (radiance, variance, entropie, φ-ratio...)
- Labels de vulnérabilités : LILITH, SUTURE, FIBONACCI, SOUVERAINETÉ

In [ ]:
# Collecter des vecteurs depuis les exemples du projet
harvest = HarvestEngine(sortie="/tmp/phi_harvest_demo.jsonl")

fichiers_source = []
for root, dirs, files in os.walk(os.path.join("..", "phi_complexity")):
    for f in files:
        if f.endswith(".py") and not f.startswith("__"):
            fichiers_source.append(os.path.join(root, f))

print(f"Fichiers Python trouv\u00e9s : {len(fichiers_source)}")

vecteurs = []
for f in fichiers_source:
    try:
        v = harvest.collecter(f)
        vecteurs.append(v)
        print(f"  \u2713 {os.path.basename(f):25s} radiance={v['radiance']:.1f}  "
              f"LILITH={v['labels']['LILITH']}  SUTURE={v['labels']['SUTURE']}")
    except Exception as e:
        print(f"  \u2717 {os.path.basename(f):25s} erreur: {e}")

print(f"\nTotal vecteurs collect\u00e9s : {len(vecteurs)}")

## 2. Exploration des Vecteurs φ

Examinons la distribution des métriques dans notre corpus.

In [ ]:
# Extraction des features
radiancies = [v["radiance"] for v in vecteurs]
variances = [v["lilith_variance"] for v in vecteurs]
entropies = [v["shannon_entropy"] for v in vecteurs]
phi_ratios = [v["phi_ratio"] for v in vecteurs]
zeta_scores = [v["zeta_score"] for v in vecteurs]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Histogramme Radiance
axes[0, 0].hist(radiancies, bins=15, color="#FFD700", edgecolor="black", alpha=0.8)
axes[0, 0].axvline(x=85, color="green", linestyle="--", label="Herm\u00e9tique (85)")
axes[0, 0].axvline(x=60, color="orange", linestyle="--", label="En \u00c9veil (60)")
axes[0, 0].set_title("Distribution de la Radiance")
axes[0, 0].legend(fontsize=8)

# Histogramme Variance
axes[0, 1].hist(variances, bins=15, color="#FF6347", edgecolor="black", alpha=0.8)
axes[0, 1].set_title("Variance de Lilith (\u03c3\u00b2_L)")

# Histogramme Entropie
axes[0, 2].hist(entropies, bins=15, color="#1E90FF", edgecolor="black", alpha=0.8)
axes[0, 2].set_title("Entropie de Shannon")

# Scatter \u03c6-Ratio vs Radiance
axes[1, 0].scatter(phi_ratios, radiancies, c=radiancies, cmap="magma", s=60, edgecolors="black")
axes[1, 0].axvline(x=PHI, color="gold", linestyle="--", label=f"\u03c6 = {PHI:.3f}")
axes[1, 0].set_xlabel("\u03c6-Ratio")
axes[1, 0].set_ylabel("Radiance")
axes[1, 0].set_title("\u03c6-Ratio vs Radiance")
axes[1, 0].legend()

# Scatter Variance vs Entropie
axes[1, 1].scatter(variances, entropies, c=radiancies, cmap="magma", s=60, edgecolors="black")
axes[1, 1].set_xlabel("Variance de Lilith")
axes[1, 1].set_ylabel("Entropie de Shannon")
axes[1, 1].set_title("Variance vs Entropie")

# Zeta scores
axes[1, 2].bar(range(len(zeta_scores)), sorted(zeta_scores, reverse=True),
               color="#9370DB", edgecolor="black", alpha=0.8)
axes[1, 2].axhline(y=PHI_INV**2, color="red", linestyle="--", label=f"Plancher \u03b6 = {PHI_INV**2:.3f}")
axes[1, 2].set_title("Zeta Scores (tri\u00e9s)")
axes[1, 2].legend(fontsize=8)

plt.suptitle("Exploration du Corpus \u03c6-Harvest", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Clustering des Vecteurs φ

Appliquons un clustering k-means simplifié (implémentation pure numpy) sur les vecteurs de résonance φ.

In [ ]:
# Extraction des vecteurs de r\u00e9sonance \u03c6 (5 dimensions)
X = np.array([v["vecteur_phi"] for v in vecteurs])
print(f"Matrice de features : {X.shape}")

# K-means simplifi\u00e9 (pure numpy, sans sklearn)
def kmeans_simple(X, k=3, max_iter=100):
    """K-means avec initialisation al\u00e9atoire."""
    np.random.seed(42)
    n = X.shape[0]
    indices = np.random.choice(n, k, replace=False)
    centroids = X[indices].copy()

    for _ in range(max_iter):
        # Assignation
        distances = np.array([[np.linalg.norm(x - c) for c in centroids] for x in X])
        labels = np.argmin(distances, axis=1)

        # Mise \u00e0 jour des centro\u00efdes
        new_centroids = np.array([X[labels == i].mean(axis=0) if np.any(labels == i)
                                  else centroids[i] for i in range(k)])

        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids

    return labels, centroids

labels, centroids = kmeans_simple(X, k=3)

# Visualisation 2D (projection sur les 2 premi\u00e8res dimensions)
fig, ax = plt.subplots(figsize=(10, 8))
colors_cluster = ["#FFD700", "#1E90FF", "#FF6347"]
cluster_names = ["Cluster Dor\u00e9", "Cluster Bleu", "Cluster Rouge"]

for i in range(3):
    mask = labels == i
    ax.scatter(X[mask, 0], X[mask, 1], c=colors_cluster[i], s=100,
               label=f"{cluster_names[i]} ({mask.sum()} fichiers)",
               edgecolors="black", linewidth=0.5)

# Centro\u00efdes
for i, c in enumerate(centroids):
    ax.plot(c[0], c[1], "*", color=colors_cluster[i], markersize=20,
            markeredgecolor="black", markeredgewidth=1.5)

ax.set_xlabel("Radiance normalis\u00e9e (dim 0)", fontsize=12)
ax.set_ylabel("Variance normalis\u00e9e (dim 1)", fontsize=12)
ax.set_title("K-Means Clustering des Vecteurs \u03c6\n(Projection 2D)", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nCentro\u00efdes des clusters :")
dims = ["radiance", "lilith", "fib_entropy", "phi_delta", "zeta"]
for i, c in enumerate(centroids):
    print(f"  {cluster_names[i]}:")
    for j, d in enumerate(dims):
        print(f"    {d:12s} = {c[j]:.4f}")

## 4. Classification des Vulnérabilités

Utilisons un classifieur par plus proches voisins (k-NN, pure numpy) pour prédire les labels de vulnérabilités.

In [ ]:
# Pr\u00e9paration des labels binaires (vuln\u00e9rable = au moins 1 label non nul)
y = np.array([1 if (v["labels"]["LILITH"] + v["labels"]["SUTURE"] +
                     v["labels"]["FIBONACCI"] + v["labels"]["SOUVERAINETE"]) > 0
              else 0 for v in vecteurs])

print("Distribution des labels :")
print(f"  Vuln\u00e9rable     : {y.sum()} ({y.sum()/len(y)*100:.1f}%)")
print(f"  Non-vuln\u00e9rable : {(1-y).sum()} ({(1-y).sum()/len(y)*100:.1f}%)")

# k-NN simplifi\u00e9 (leave-one-out cross-validation)
def knn_predict(X, y, k=3):
    """k-NN avec validation leave-one-out."""
    n = len(X)
    predictions = np.zeros(n, dtype=int)
    for i in range(n):
        distances = np.array([np.linalg.norm(X[i] - X[j]) for j in range(n) if j != i])
        y_others = np.array([y[j] for j in range(n) if j != i])
        nearest = np.argsort(distances)[:k]
        predictions[i] = 1 if y_others[nearest].sum() > k // 2 else 0
    return predictions

predictions = knn_predict(X, y, k=3)
accuracy = (predictions == y).mean()

print(f"\n  Accuracy (LOO-CV, k=3) : {accuracy:.2%}")

# Scatter des pr\u00e9dictions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors_true = ["#32CD32" if yi == 0 else "#FF6347" for yi in y]
ax1.scatter(X[:, 0], X[:, 1], c=colors_true, s=80, edgecolors="black", linewidth=0.5)
ax1.set_title("Labels R\u00e9els\n(Vert=Sain, Rouge=Vuln\u00e9rable)", fontsize=12)
ax1.set_xlabel("Radiance normalis\u00e9e")
ax1.set_ylabel("Variance normalis\u00e9e")

colors_pred = ["#32CD32" if pi == 0 else "#FF6347" for pi in predictions]
ax2.scatter(X[:, 0], X[:, 1], c=colors_pred, s=80, edgecolors="black", linewidth=0.5)
ax2.set_title(f"Pr\u00e9dictions k-NN (k=3)\nAccuracy={accuracy:.2%}", fontsize=12)
ax2.set_xlabel("Radiance normalis\u00e9e")
ax2.set_ylabel("Variance normalis\u00e9e")

plt.suptitle("Classification de Vuln\u00e9rabilit\u00e9s \u2014 Vecteurs \u03c6", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Visualisation t-SNE (Simplifiée)

Réduction de dimensionnalité pour visualiser l’espace φ en 2D.

In [ ]:
# t-SNE simplifi\u00e9 (approximation par MDS \u2014 Multi-Dimensional Scaling)
def mds_simple(X, n_components=2):
    """MDS simplifi\u00e9 (pure numpy) pour visualisation 2D."""
    n = X.shape[0]
    # Matrice de distances
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = np.linalg.norm(X[i] - X[j])

    # Double centrage
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ (D ** 2) @ H

    # D\u00e9composition en valeurs propres
    eigenvalues, eigenvectors = np.linalg.eigh(B)
    idx = np.argsort(eigenvalues)[::-1][:n_components]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Projection
    coords = eigenvectors * np.sqrt(np.maximum(eigenvalues, 0))
    return coords

coords_2d = mds_simple(X)

fig, ax = plt.subplots(figsize=(10, 8))

# Colorer par radiance
scatter = ax.scatter(coords_2d[:, 0], coords_2d[:, 1],
                     c=[v["radiance"] for v in vecteurs],
                     cmap="magma", s=100, edgecolors="black", linewidth=0.5)
plt.colorbar(scatter, label="Radiance", ax=ax)

# Marquer les fichiers vuln\u00e9rables
for i, v in enumerate(vecteurs):
    total_vuln = sum(v["labels"].values())
    if total_vuln > 0:
        ax.annotate("\u26a0", (coords_2d[i, 0], coords_2d[i, 1]),
                   fontsize=14, ha="center", va="center", color="red")

ax.set_xlabel("Dimension 1", fontsize=12)
ax.set_ylabel("Dimension 2", fontsize=12)
ax.set_title("Espace \u03c6 \u2014 Projection MDS\n(\u26a0 = vuln\u00e9rabilit\u00e9s d\u00e9tect\u00e9es)", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Conclusion — ML Souverain

| Aspect | Approche φ-Harvest |
|---|---|
| **Privacy** | Zéro identifiant, zéro code source exporté |
| **Features** | Vecteur φ normalisé (5 dimensions) |
| **Labels** | LILITH, SUTURE, FIBONACCI, SOUVERAINETÉ |
| **Format** | JSONL incrémental, prêt pour l’entraînement |
| **Dépendances** | Zéro (numpy optionnel pour la visualisation) |

> *“L’IA souveraine n’a pas besoin de voir votre code pour comprendre sa géométrie.”*

---
*Ancré dans la Bibliothèque Céleste — Morphic Phi Framework — 2026*